# Visualization of trained prior mean model

In [ ]:
import sys
import os
import warnings

import numpy as np
import torch
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))

warnings.filterwarnings("ignore")

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SLURRIES = ["G50", "G45", "G40", "G40+IPA"]

## Model parameters

In [ ]:
data = torch.load("../../models/v1/feature_models/prior_mean.pt", map_location=device)
data["model"]["state_dict"]["params.H_params"]

## 1D plot

In [ ]:
X = pd.read_csv("../../_temp/v1/X.csv", index_col=[0, 1, 2])
y = pd.read_csv("../../_temp/v1/y.csv", index_col=[0, 1, 2])
Xpred = pd.read_csv("../../_temp/v1/Xpred_1D.csv", index_col=[0, 1, 2])

In [ ]:
columns = ["slurry", "cosine_of_contact_angle"]
cos_thetas = X["cosine_of_contact_angle"].reset_index()[columns]
slurry_map = cos_thetas.drop_duplicates().set_index("cosine_of_contact_angle")["slurry"]

slurries = Xpred["cosine_of_contact_angle"].map(slurry_map)
unique_slurries = [s for s in SLURRIES if s in slurries.unique()]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

Rgt_pred = Xpred["gap_to_thickness_ratio"].unique()
Cas = Xpred["capillary_number"].unique()

Cas_sorted = np.sort(Cas)
cmap = plt.get_cmap("viridis", len(Cas_sorted))
norm = mcolors.BoundaryNorm(
    np.concatenate(
        [
            [Cas_sorted[0] * 0.9],
            (Cas_sorted[:-1] + Cas_sorted[1:]) / 2,
            [Cas_sorted[-1] * 1.1],
        ]
    ),
    ncolors=len(Cas_sorted),
)

In [ ]:
pred_mean = pd.read_csv("../../benchmarks/v1/prior_mean.Xpred_1D.csv").pivot(
    index=["index", "batch"], columns="target", values="value"
)

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey="row")

for slurry, ax in zip(unique_slurries, axes):
    ok = X.index.get_level_values("slurry") == slurry
    this_X = X[ok]
    this_H = y[ok]["H"].values

    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred]
    this_mean = pred_mean["H"][ok_pred.values]

    for ca in this_X["capillary_number"].unique():
        ok = this_X["capillary_number"] == ca
        ok_pred = this_Xpred["capillary_number"] == ca

        x_obs = this_X[ok]["gap_to_thickness_ratio"].to_numpy()
        y_obs = this_H[ok]
        x_values = np.sort(np.unique(x_obs))
        box_width = 0.6 * np.min(np.diff(x_values)) if len(x_values) > 1 else 0.1
        color = cmap(norm(ca))
        ax.boxplot(
            [y_obs[x_obs == x0] for x0 in x_values],
            positions=x_values,
            widths=box_width,
            whis=(2.5, 97.5),
            patch_artist=True,
            manage_ticks=False,
            boxprops={"facecolor": color, "edgecolor": color, "alpha": 0.65},
            whiskerprops={"color": color},
            capprops={"color": color},
            medianprops={"color": color},
            flierprops={
                "marker": "o",
                "markerfacecolor": color,
                "markeredgecolor": color,
                "markersize": 2,
            },
        )

        ax.plot(
            this_Xpred[ok_pred]["gap_to_thickness_ratio"],
            this_mean[ok_pred.values],
            color=cmap(norm(ca)),
        )

    (cos_theta,) = this_X["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("H")

fig.suptitle("Prior mean")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey="row")

for slurry, ax in zip(unique_slurries, axes):
    ok = X.index.get_level_values("slurry") == slurry
    this_X = X[ok]
    this_phi = y[ok]["phi_1"].values

    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred]
    this_mean = pred_mean["phi_1"][ok_pred.values]

    for ca in this_X["capillary_number"].unique():
        ok = this_X["capillary_number"] == ca
        ok_pred = this_Xpred["capillary_number"] == ca

        x_obs = this_X[ok]["gap_to_thickness_ratio"].to_numpy()
        y_obs = this_phi[ok]
        x_values = np.sort(np.unique(x_obs))
        box_width = 0.6 * np.min(np.diff(x_values)) if len(x_values) > 1 else 0.1
        color = cmap(norm(ca))
        ax.boxplot(
            [y_obs[x_obs == x0] for x0 in x_values],
            positions=x_values,
            widths=box_width,
            whis=(2.5, 97.5),
            patch_artist=True,
            manage_ticks=False,
            boxprops={"facecolor": color, "edgecolor": color, "alpha": 0.65},
            whiskerprops={"color": color},
            capprops={"color": color},
            medianprops={"color": color},
            flierprops={
                "marker": "o",
                "markerfacecolor": color,
                "markeredgecolor": color,
                "markersize": 2,
            },
        )

        ax.plot(
            this_Xpred[ok_pred]["gap_to_thickness_ratio"],
            this_mean[ok_pred.values],
            color=cmap(norm(ca)),
        )

    (cos_theta,) = this_X["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("phi1")

fig.suptitle("Prior mean")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(slurry_map), sharex=True, sharey="row")

for slurry, ax in zip(unique_slurries, axes):
    ok = X.index.get_level_values("slurry") == slurry
    this_X = X[ok]
    this_phi = y[ok]["phi_3"].values

    ok_pred = slurries == slurry
    this_Xpred = Xpred[ok_pred]
    this_mean = pred_mean["phi_3"][ok_pred.values]

    for ca in this_X["capillary_number"].unique():
        ok = this_X["capillary_number"] == ca
        ok_pred = this_Xpred["capillary_number"] == ca

        x_obs = this_X[ok]["gap_to_thickness_ratio"].to_numpy()
        y_obs = this_phi[ok]
        x_values = np.sort(np.unique(x_obs))
        box_width = 0.6 * np.min(np.diff(x_values)) if len(x_values) > 1 else 0.1
        color = cmap(norm(ca))
        ax.boxplot(
            [y_obs[x_obs == x0] for x0 in x_values],
            positions=x_values,
            widths=box_width,
            whis=(2.5, 97.5),
            patch_artist=True,
            manage_ticks=False,
            boxprops={"facecolor": color, "edgecolor": color, "alpha": 0.65},
            whiskerprops={"color": color},
            capprops={"color": color},
            medianprops={"color": color},
            flierprops={
                "marker": "o",
                "markerfacecolor": color,
                "markeredgecolor": color,
                "markersize": 2,
            },
        )

        ax.plot(
            this_Xpred[ok_pred]["gap_to_thickness_ratio"],
            this_mean[ok_pred.values],
            color=cmap(norm(ca)),
        )

    (cos_theta,) = this_X["cosine_of_contact_angle"].unique()
    ax.set_title(f"Cos θ={cos_theta:.2f}")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
)
cbar.set_label("Ca")
quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

fig.supxlabel("Rgt")
fig.supylabel("phi3")

fig.suptitle("Prior mean")
plt.show()